# Mono Lake Water-Balance Model
### Environmental modeling of a terminal lake

Mono Lake is a saline terminal lake in California. This model reproduces the water-balance model from Ford's *Modelling the Environment*. It tracks the lake's volume and elevation as it gains water through precipitation and runoff and loses it through evaporation and human water export (diversions).

## How it works

The model uses cubic splines to map the lake's volume to its surface area and elevation. The volume changes based on the net balance of inflows and outflows:

| Component | Description |
|:---|:---|
| Inflows | Precipitation (on surface), Runoff, Other inputs |
| Outflows | Evaporation (from surface), Export (diversions), Other outputs |

Because evaporation and precipitation depend on the lake's surface area, which in turn depends on its volume, there is a non-linear feedback loop.

## Imports

In [ ]:
from dissmodel.core import Environment
from dissmodel_sysdyn.models import MonoLake
from dissmodel.visualization import Chart

## ⚠️ Important: instantiation order

The `Environment` must always be created **before** any model.

```
Environment  →  MonoLake  →  Chart
```

In [ ]:
env = Environment()

## Setting up the model

The Mono Lake model accepts the following parameters:

| Parameter | Description | Default |
|:---|:---|:---|
| `water_in_lake` | Initial volume (kilo-acre-feet) | 2228.0 |
| `export` | Annual water export (KAF/year) | 100.0 |
| `prec_rate` | Precipitation rate (feet/year) | 0.67 |
| `evap_rate` | Evaporation rate (feet/year) | 3.75 |

In [ ]:
import numpy as np
from scipy.interpolate import CubicSpline
from dissmodel.core import Model
from dissmodel.visualization import track_plot

_VOLUME_KAF  = np.array([0, 1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000], dtype=float)
_SURFACE_KM2 = np.array([0, 24.7, 35.3, 48.6, 54.3, 57.2, 61.6, 66.0, 69.9], dtype=float)
_ELEV_FT     = np.array([6224, 6335, 6369, 6392, 6412, 6430, 6447, 6463, 6477], dtype=float)

_water_surface   = CubicSpline(_VOLUME_KAF, _SURFACE_KM2, extrapolate=True)
_water_elevation = CubicSpline(_VOLUME_KAF, _ELEV_FT,     extrapolate=True)

@track_plot("level", "blue")
@track_plot("water_in_lake", "cyan")
class MonoLake(Model):
    def setup(self, water_in_lake=2228.0, level=6375.0, prec_rate=0.67, runoff=150.0, other_in=47.6, evap_rate=3.75, other_out=33.6, export=100.0) -> None:
        self.water_in_lake, self.level = water_in_lake, level
        self.prec_rate, self.runoff, self.other_in = prec_rate, runoff, other_in
        self.evap_rate, self.other_out, self.export = evap_rate, other_out, export

    def execute(self) -> None:
        surface = float(_water_surface(self.water_in_lake))
        total_in = surface * self.prec_rate + self.runoff + self.other_in
        total_out = surface * self.evap_rate + self.export + self.other_out
        self.water_in_lake += total_in - total_out
        self.level = float(_water_elevation(self.water_in_lake))

In [ ]:
MonoLake(export=100.0)

## Visualization

The chart will show both the lake's volume and its elevation over time.

In [ ]:
Chart(show_legend=True, show_grid=True, title="Mono Lake Water Balance")

## Running the simulation

In [ ]:
env.run(50)

## Guided Experiments

- **Diversion Cutoff**: Set `export=0` to see how the lake recovers when water exports are stopped.
- **Climate Change**: Increase `evap_rate` or decrease `prec_rate` to simulate a drying climate.
- **Sustainability**: Find the `export` value that keeps the lake level stable at its current elevation.

## Conclusion

The Mono Lake model demonstrates how system dynamics can be used to inform environmental policy. By understanding the balance of natural and human-driven processes, we can better manage precious water resources.